## Quadruped robot vision

This project aims to **bring vision to quadruped robots**.

Since quadropeds have near-ground point of view, the object detector was designed to overcome those obsticals. The choice of having the detection of dogs and humans was made to minic the dog behavor of finding their "race" and "best friends". This should allow robots to know the right way of moving.

To achive this, a YOLO with the COCO and Standrod Dogs dataset were chosen.

**YOLO (You Only Look Once)** was selected as it is a popular, highly efficient, real-time object detection algorithm, which uses a convolutional neural network (CNN) with a signal pass of processing for a given image.
**COCO (Common Objects in Context)** is a large-scale dataset that contains images of many objects with annotations to them.
**Stanford Dogs** is a wide dataset of dogs images of different breads.

However, the datasets do not provide the right training data for the purpose, so a new custom dataset was developed based on those two. They randomly crop parts of the images to simulate low-angle views.

The best training was run for **23.890 hours** on a CPU completing 6 epochs.

In [ ]:
# Main imports
import os
import json
import random
import xml.etree.ElementTree as ET # Stanford Dogs XML
import cv2
import numpy as np
from tqdm import tqdm # Nice progress bar
from ultralytics import YOLO
import torch

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Main paths
COCO_IMAGES = "coco/train2017"
COCO_ANN = "coco/annotations/instances_train2017.json"

DOG_ROOT = "stanford-dogs-dataset"
DOG_IMG_DIR = DOG_ROOT + "/images/Images"
DOG_ANN_DIR = DOG_ROOT + "/annotations/Annotation"

# Custom dataset, which is balanced for humans and dogs
OUTPUT_ROOT = "dataset_balanced"
IMG_OUT = OUTPUT_ROOT + "/images"
LBL_OUT = OUTPUT_ROOT + "/labels"

for p in [OUTPUT_ROOT, IMG_OUT, LBL_OUT]:
    os.makedirs(p, exist_ok=True)

In [ ]:
# Convert (normalize) bounding box coordinates into YOLO's format
def save_yolo_label(label_path, cls, xmin, ymin, xmax, ymax, img_w, img_h):
    x_center = ((xmin + xmax) / 2) / img_w
    y_center = ((ymin + ymax) / 2) / img_h
    w = (xmax - xmin) / img_w
    h = (ymax - ymin) / img_h

    with open(label_path, "w") as f:
        f.write(f"{cls} {x_center} {y_center} {w} {h}\n")

In [ ]:
print("Processing humans...")

with open(COCO_ANN, "r") as f:
    coco = json.load(f)

images_dict = {img["id"]: img for img in coco["images"]}

count = 0

for ann in tqdm(coco["annotations"]):
    if ann["category_id"] != 1:  # COCO "person"
        continue

    img_info = images_dict[ann["image_id"]]
    img_path = os.path.join(COCO_IMAGES, img_info["file_name"])

    if not os.path.exists(img_path):
        continue

    x, y, w, h = ann["bbox"]
    x = int(x)
    y = int(y)
    w = int(w)
    h = int(h)

    img_cv = cv2.imread(img_path)
    if img_cv is None:
        continue
    h_img, w_img = img_cv.shape[:2]

    # crop range 40–70% of height
    ratio = random.uniform(0.4, 0.7)
    lower_y = int(y + h * (1 - ratio))

    xmin = x
    ymin = lower_y
    xmax = x + w
    ymax = y + h

    cropped = img_cv[ymin:ymax, xmin:xmax]

    out_img_name = f"human_{count}.jpg"
    out_lbl_name = f"human_{count}.txt"

    cv2.imwrite(os.path.join(IMG_OUT, out_img_name), cropped)

    # save YOLO label: entire crop is one bounding box
    save_yolo_label(
        os.path.join(LBL_OUT, out_lbl_name),
        cls=0,  # human
        xmin=0,
        ymin=0,
        xmax=(xmax - xmin),
        ymax=(ymax - ymin),
        img_w=(xmax - xmin),
        img_h=(ymax - ymin)
    )

    count += 1

print("Human samples:", count)
HUMAN_COUNT = count

Processing humans...


FileNotFoundError: [Errno 2] No such file or directory: 'coco/annotations/instances_val2017.json'

In [ ]:
print("Processing dogs...")

files_list = []
for root_dir, _, files in os.walk(DOG_ANN_DIR):
    for f in files:
        files_list.append((root_dir, f))

dog_count = 0

for root_dir, ann_file in tqdm(files_list):
    xml_path = os.path.join(root_dir, ann_file)
    tree = ET.parse(xml_path)
    root = tree.getroot()

    img_path = xml_path.replace("annotations", "images").replace("Annotation", "Images") + ".jpg"

    if not os.path.exists(img_path):
        continue

    img_cv = cv2.imread(img_path)
    h_img, w_img = img_cv.shape[:2]

    obj = root.find("object")
    box = obj.find("bndbox")

    xmin = int(box.find("xmin").text)
    ymin = int(box.find("ymin").text)
    xmax = int(box.find("xmax").text)
    ymax = int(box.find("ymax").text)

    # Full dog
    dog_crop = img_cv[ymin:ymax, xmin:xmax]

    out_img_name = f"dog_full_{dog_count}.jpg"
    out_lbl_name = f"dog_full_{dog_count}.txt"

    cv2.imwrite(os.path.join(IMG_OUT, out_img_name), dog_crop)

    # YOLO format: full crop
    save_yolo_label(
        os.path.join(LBL_OUT, out_lbl_name),
        cls=1,
        xmin=0,
        ymin=0,
        xmax=(xmax - xmin),
        ymax=(ymax - ymin),
        img_w=(xmax - xmin),
        img_h=(ymax - ymin)
    )

    dog_count += 1

    # Dog's lower part
    crop_ratio = random.uniform(0.6, 0.8)
    new_ymin = int(ymax - (ymax - ymin) * crop_ratio)

    lb_crop = img_cv[new_ymin:ymax, xmin:xmax]

    out_img_name = f"dog_lower_{dog_count}.jpg"
    out_lbl_name = f"dog_lower_{dog_count}.txt"

    cv2.imwrite(os.path.join(IMG_OUT, out_img_name), lb_crop)

    save_yolo_label(
        os.path.join(LBL_OUT, out_lbl_name),
        cls=1,
        xmin=0,
        ymin=0,
        xmax=(xmax - xmin),
        ymax=(ymax - new_ymin),
        img_w=(xmax - xmin),
        img_h=(ymax - new_ymin)
    )

    dog_count += 1

print("Dog samples:", dog_count)
DOG_COUNT = dog_count

Processing dogs...


100%|██████████| 20580/20580 [39:29<00:00,  8.69it/s]

Dog samples: 41160


In [ ]:
print("Balancing dataset...")

target = max(HUMAN_COUNT, DOG_COUNT)

def simple_augment(img_cv):
    h, w = img_cv.shape[:2]
    xmin, ymin, xmax, ymax = 0, 0, w, h

    if random.random() < 0.7: # Random brightness
        beta = int((random.random() - 0.5) * 100)
        img_cv = cv2.convertScaleAbs(img_cv, alpha=1.0, beta=beta)

    if random.random() < 0.7: # Random contrast
        factor = 0.7 + random.random() * 0.6
        img_cv = cv2.convertScaleAbs(img_cv, alpha=factor, beta=0)

    if random.random() < 0.3: # Random blur
        k = random.uniform(0.3, 1.2)
        ksize = int(max(3, int(k * 3)))
        if ksize % 2 == 0:
            ksize += 1
        img_cv = cv2.GaussianBlur(img_cv, (ksize, ksize), 0)

    if random.random() < 0.6: # Random scaling
        scale = random.uniform(0.8, 1.2)
        new_w, new_h = int(w * scale), int(h * scale)
        resized = cv2.resize(img_cv, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

        if scale >= 1.0:
            # Center-crop back to original size
            left = (new_w - w) // 2
            top = (new_h - h) // 2
            img_cv = resized[top:top + h, left:left + w]
            # bbox stays full frame
        else:
            # Center-pad back to original size
            pad_left = (w - new_w) // 2
            pad_top = (h - new_h) // 2
            padded = np.zeros((h, w, 3), dtype=np.uint8)
            padded[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized
            img_cv = padded
            xmin = pad_left
            ymin = pad_top
            xmax = pad_left + new_w
            ymax = pad_top + new_h

    return img_cv, (xmin, ymin, xmax, ymax)


def augment_class(cls_id, prefix):
    src_imgs = [f for f in os.listdir(IMG_OUT) if f.startswith(prefix)]
    src_lbls = [f for f in os.listdir(LBL_OUT) if f.startswith(prefix)]

    img_label_pairs = list(zip(src_imgs, src_lbls))
    count = len(img_label_pairs)

    with tqdm(total=target, initial=count, desc=f"{prefix} augment") as pbar:
        while count < target:
            img_name, lbl_name = random.choice(img_label_pairs)

            img_path = os.path.join(IMG_OUT, img_name)
            label_path = os.path.join(LBL_OUT, lbl_name)

            img_cv = cv2.imread(img_path)
            h, w = img_cv.shape[:2]

            aug_img_cv, (xmin, ymin, xmax, ymax) = simple_augment(img_cv)

            new_img_name = f"{prefix}_aug_{count}.jpg"
            new_lbl_name = f"{prefix}_aug_{count}.txt"
            new_img_path = os.path.join(IMG_OUT, new_img_name)
            new_lbl_path = os.path.join(LBL_OUT, new_lbl_name)

            cv2.imwrite(new_img_path, aug_img_cv)

            save_yolo_label(
                new_lbl_path,
                cls=int(cls_id),
                xmin=xmin,
                ymin=ymin,
                xmax=xmax,
                ymax=ymax,
                img_w=w,
                img_h=h
            )

            count += 1
            pbar.update(1)

    return count


human_final = augment_class(0, "human")
dog_final = augment_class(1, "dog")

print("Final counts:", human_final, dog_final)

Balancing dataset...


human augment:  27%|██▋       | 11235/41160 [00:45<1:39:18,  5.02it/s]


KeyboardInterrupt: 

In [ ]:
print("Creating train/val split...")

all_imgs = [f for f in os.listdir(IMG_OUT) if f.endswith(".jpg")]
random.shuffle(all_imgs)

split = int(0.8 * len(all_imgs))
train_files = all_imgs[:split]
val_files = all_imgs[split:]

# os.mv() and opening and saving files took a lot of time,
#  so using files with the paths to the dataset images was
#  an optimization, which is supported by YOLO
train_list_path = OUTPUT_ROOT + "/train.txt"
val_list_path = OUTPUT_ROOT + "/val.txt"

with open(train_list_path, "w") as f:
    for img in train_files:
        f.write(os.path.join(IMG_OUT, img) + "\n")

with open(val_list_path, "w") as f:
    for img in val_files:
        f.write(os.path.join(IMG_OUT, img) + "\n")

print("Done.")

In [ ]:
# YOLO YAML file
yaml = f"""
path: {os.path.abspath(OUTPUT_ROOT)}
train: train.txt
val: val.txt

names:
  0: human
  1: dog

"""

with open("dataset.yaml", "w") as f:
    f.write(yaml)

print("Created dataset.yaml")

Created dataset.yaml


In [ ]:
# Load semi-pretrained model
model = YOLO("yolov8n.pt")

model.train(
    data="dataset.yaml",
    epochs=50,
    imgsz=320,
    batch=8,
    patience=10 # Epochs before stopping
)

New https://pypi.org/project/ultralytics/8.3.230 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.70 🚀 Python-3.11.13 torch-2.5.1 CPU (Apple M2 Pro)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=dataset.yaml, epochs=6, time=None, patience=10, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train5, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_co

train: Scanning dataset_balanced/labels... 40696 images, 0 backgrounds, 1152 corrupt: 100%|██████████| 41848/41848 [01:30<00:00, 460.74it/s]

train: WARNING ⚠️ dataset_balanced/images/human_1013.jpg: ignoring corrupt image/label: image size (8, 11) <10 pixels
train: WARNING ⚠️ dataset_balanced/images/human_1016.jpg: ignoring corrupt image/label: image size (16, 9) <10 pixels
train: WARNING ⚠️ dataset_balanced/images/human_10244.jpg: ignoring corrupt image/label: image size (16, 6) <10 pixels
train: WARNING ⚠️ dataset_balanced/images/human_10263.jpg: ignoring corrupt image/label: image size (16, 9) <10 pixels
train: WARNING ⚠️ dataset_balanced/images/human_10264.jpg: ignoring corrupt image/label: image size (9, 10) <10 pixels
train: WARNING ⚠️ dataset_balanced/images/human_1033.jpg: ignoring corrupt image/label: image size (11, 6) <10 pixels
train: WARNING ⚠️ dataset_balanced/images/human_10333.jpg: ignoring corrupt image/label: image size (17, 9) <10 pixels
train: WARNING ⚠️ dataset_balanced/images/human_10339.jpg: ignoring corrupt image/label: image size (15, 8) <10 pixels
train: WARNING ⚠️ dataset_balanced/images/human_103

train: New cache created: dataset_balanced/labels.cache


val: Scanning dataset_balanced/labels... 10173 images, 0 backgrounds, 289 corrupt: 100%|██████████| 10462/10462 [00:16<00:00, 622.93it/s]

...

                   all      10173      10173      0.983      0.978      0.993      0.993



6 epochs completed in 23.890 hours.
Optimizer stripped from /opt/homebrew/runs/detect/train5/weights/last.pt, 6.2MB
Optimizer stripped from /opt/homebrew/runs/detect/train5/weights/best.pt, 6.2MB

Validating /opt/homebrew/runs/detect/train5/weights/best.pt...
Ultralytics 8.3.70 🚀 Python-3.11.13 torch-2.5.1 CPU (Apple M2 Pro)
Model summary (fused): 168 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 636/636 [23:24<00:00,  2.21s/it]


                   all      10173      10173      0.983      0.978      0.993      0.993
                 human       1968       1968      0.973      0.964      0.991      0.991
                   dog       8205       8205      0.993      0.991      0.995      0.995
Speed: 0.2ms preprocess, 132.9ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /opt/homebrew/runs/detect/train5


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x155505750>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Validate
with open("dataset_balanced/val.txt") as f:
    val_files = [line.strip() for line in f]

example_path = random.choice(val_files)

print("Testing on:", example_path)

results = model.predict(source=example_path, imgsz=320, save=True)

Testing on: dataset_balanced/images/dog_full_7240.jpg

image 1/1 /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dataset_balanced/images/dog_full_7240.jpg: 640x608 1 dog, 56.9ms
Speed: 2.5ms preprocess, 56.9ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 608)
Results saved to /opt/homebrew/runs/detect/train54


In [ ]:
model = YOLO("/opt/homebrew/runs/detect/train5/weights/best.pt")

test_image = "/Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dog.jpg"

print("Testing on:", test_image)

results = model.predict(source=test_image, imgsz=320, save=True)

Testing on: /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dog.jpg

image 1/1 /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dog.jpg: 640x640 1 dog, 78.2ms
Speed: 11.0ms preprocess, 78.2ms inference, 10.3ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /opt/homebrew/runs/detect/predict


In [ ]:
test_image = "/Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/human lower body.png"

print("Testing on:", test_image)

results = model.predict(source=test_image, imgsz=320, save=True)

Testing on: /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/human lower body.png

image 1/1 /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/human lower body.png: 544x640 1 human, 59.5ms
Speed: 7.7ms preprocess, 59.5ms inference, 0.3ms postprocess per image at shape (1, 3, 544, 640)
Results saved to /opt/homebrew/runs/detect/predict


In [ ]:
test_image = "/Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dog lower body.jpg"

print("Testing on:", test_image)

results = model.predict(source=test_image, imgsz=320, save=True)

Testing on: /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dog lower body.jpg

image 1/1 /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/dog lower body.jpg: 512x640 1 dog, 39.6ms
Speed: 2.0ms preprocess, 39.6ms inference, 0.6ms postprocess per image at shape (1, 3, 512, 640)
Results saved to /opt/homebrew/runs/detect/predict


In [ ]:
test_image = "/Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/balls.jpg"

print("Testing on:", test_image)

results = model.predict(source=test_image, imgsz=320, save=True)

Testing on: /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/balls.jpg

image 1/1 /Volumes/ssd256/Study/Singidunum Univeristy/Computer Vision/balls.jpg: 640x640 1 human, 88.1ms
Speed: 7.7ms preprocess, 88.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /opt/homebrew/runs/detect/predict
